In [21]:
import cv2 
import numpy as np
from IPython.display import HTML, display
import os

os.makedirs("video/output", exist_ok=True)

In [22]:
def show_videos(*items, min_width=200):
    def box(path, title):
        media = f'<video style="width:100%;" autoplay loop muted playsinline><source src="{path}" type="video/mp4"></video>'
        return f"""
        <div style="flex:1; min-width:{min_width}px; padding:10px; background:white; font-family:sans-serif; text-align:center; margin:5px; box-sizing:border-box;">
            <p style="margin:0 0 8px 0; font-size:14px; color:#333;">{title}</p>
            {media}
        </div>"""
    display(HTML(f'<div style="display:flex; flex-wrap:nowrap; width:100%; background:white;">{"".join(box(p, t) for p, t in items)}</div>'))

In [23]:
Video_path = "video/video.mp4"
Avg_bg_video_path = "video/output/background_video_avg.mp4"

video = cv2.VideoCapture(Video_path)
fps = int(video.get(cv2.CAP_PROP_FPS))
w = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"avc1")
avg_video_out = cv2.VideoWriter(Avg_bg_video_path, fourcc, fps, (w, h), isColor=False)

frame_buffer = []
buffer_size = 10
frame_count = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
    frame_buffer.append(frame_gray)
    
    if len(frame_buffer) > buffer_size:
        frame_buffer.pop(0)
    
    # Always write a frame for every input frame
    if len(frame_buffer) == 1:
        # For first frame, just use the frame itself
        bg_avg = frame_gray.astype(np.uint8)
    else:
        # Use average of available frames
        bg_avg = np.mean(frame_buffer, axis=0).astype(np.uint8)
    
    avg_video_out.write(bg_avg)
    
    frame_count += 1

video.release()
avg_video_out.release()

print(f"Total frames processed: {frame_count}")

show_videos(
    (Video_path, "Original Video"),
    (Avg_bg_video_path, f"Average Background")
)

Total frames processed: 71


In [24]:
threshold = 25
change_video_path = "video/output/videochange_avg.mp4"

video = cv2.VideoCapture(Video_path)
bg_video = cv2.VideoCapture(Avg_bg_video_path)

out = cv2.VideoWriter(change_video_path, fourcc, fps, (w, h))

while True:
    ret, frame = video.read()
    _, bg_frame = bg_video.read()
    if not ret:
        break    
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    bg_gray = cv2.cvtColor(bg_frame, cv2.COLOR_BGR2GRAY)
    
    diff = cv2.absdiff(frame_gray, bg_gray)
    _, mask = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
    out.write(mask)

video.release()
bg_video.release()
out.release()

show_videos(
    (Video_path, "Original Video"),
    (Avg_bg_video_path, "Average Background"),
    (change_video_path, "Change Detection (Average BG)")
)

In [25]:
thresholds = [5, 10, 20, 30, 50]

# Create output directory for threshold videos
threshold_output_dir = "video/output/threshold"
os.makedirs(threshold_output_dir, exist_ok=True)

for t in thresholds:
    video = cv2.VideoCapture(Video_path)
    bg_video = cv2.VideoCapture(Avg_bg_video_path) 

    out_path = f"{threshold_output_dir}/threshold_{t}.mp4"
    out = cv2.VideoWriter(out_path, fourcc, fps, (w, h), isColor=False)

    while True:
        ret, frame = video.read()
        _, bg_frame = bg_video.read()
        if not ret:
            break    
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        if len(bg_frame.shape) == 3:
            bg_gray = cv2.cvtColor(bg_frame, cv2.COLOR_BGR2GRAY)
        else:
            bg_gray = bg_frame
        
        diff = cv2.absdiff(frame_gray, bg_gray)
        _, mask = cv2.threshold(diff, t, 255, cv2.THRESH_BINARY)
        out.write(mask)

    video.release()
    bg_video.release()
    out.release()

# Show all threshold videos
show_videos(*[(f"{threshold_output_dir}/threshold_{t}.mp4", f"Threshold = {t}") for t in thresholds])

In [26]:
# ── parameters you can tune ──────────────────────────────────────────
MIN_COMPONENT_PX = 50         # smaller blobs are cancelled
KERNEL_SIZE      = 5          # morphological kernel size (odd number)
ERODE_ITER       = 1          # erosion passes  (removes thin noise)
DILATE_ITER      = 2          # dilation passes (fills gaps, connects close blobs)
# ─────────────────────────────────────────────────────────────────────

# Open the existing video
change_video_path = "video/output/videochange_avg.mp4"
change = cv2.VideoCapture(change_video_path)

# Create kernel for morphological operations
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (KERNEL_SIZE, KERNEL_SIZE))

# Setup output video writer
cleaned_video_path = "video/output/change_cleaned.mp4"
os.makedirs(os.path.dirname(cleaned_video_path), exist_ok=True)
out_cln = cv2.VideoWriter(cleaned_video_path, cv2.VideoWriter_fourcc(*"avc1"), fps, (w, h))

while True:
    ret, frame = change.read()
    if not ret:
        break
    
    # Convert to grayscale if needed
    if len(frame.shape) == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # ── 1. cancel small connected components ─────────────────────────
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(frame, connectivity=8)
    clean = np.zeros_like(frame)
    for lbl in range(1, n_labels):                          # 0 = background
        if stats[lbl, cv2.CC_STAT_AREA] >= MIN_COMPONENT_PX:
            clean[labels == lbl] = 255
    
    # ── 2. morphological operations ──────────────────────────────────
    clean = cv2.erode(clean, kernel, iterations=ERODE_ITER)   # remove thin noise
    clean = cv2.dilate(clean, kernel, iterations=DILATE_ITER)  # smooth & connect
    
    # Write as BGR so VideoWriter is happy
    out_cln.write(cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR))

# Release resources
change.release()
out_cln.release()

show_videos(
    (Video_path, "Original Video"),
    (change_video_path, "Original Change Detection"),
    (cleaned_video_path, "Cleaned Change Detection")
)

In [27]:
# Open the cleaned video
video = cv2.VideoCapture("video/change_cleaned.mp4")

# Get video properties
fps = int(video.get(cv2.CAP_PROP_FPS))
w = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Create output video with colored blobs for visualization
out_blobs = cv2.VideoWriter("video/change_blobs_visualized.mp4", 
                            cv2.VideoWriter_fourcc(*"avc1"), 
                            fps, (w, h))

frame_count = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    frame_count += 1
    
    # Convert to grayscale if needed
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()
    
    # ── COMPUTE CONNECTED COMPONENTS (BLOBS) ────────────────────────
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        gray, connectivity=8
    )
    
    # ── VISUALIZATION ────────────────────────────────────────────────
    # Create colored visualization of blobs
    vis_frame = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    
    # Draw bounding boxes and labels for each blob (skip label 0 = background)
    for label in range(1, num_labels):
        x = stats[label, cv2.CC_STAT_LEFT]
        y = stats[label, cv2.CC_STAT_TOP]
        width = stats[label, cv2.CC_STAT_WIDTH]
        height = stats[label, cv2.CC_STAT_HEIGHT]
        area = stats[label, cv2.CC_STAT_AREA]
        cx, cy = centroids[label]
        
        # Draw bounding box (green)
        cv2.rectangle(vis_frame, (x, y), (x + width, y + height), (0, 255, 0), 2)
        
        # Draw centroid (red)
        cv2.circle(vis_frame, (int(cx), int(cy)), 3, (0, 0, 255), -1)
        
        # Add label with area
        cv2.putText(vis_frame, f"A:{area}", 
                   (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
    
    # Add frame info
    cv2.putText(vis_frame, f"Frame: {frame_count} | Blobs: {num_labels - 1}", 
               (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    out_blobs.write(vis_frame)

# Release resources
video.release()
out_blobs.release()

# ── SHOW VISUALIZATION ──────────────────────────────────────────────
show_media(
    ("video/change_cleaned.mp4", "Cleaned Change Detection"),
    ("video/change_blobs_visualized.mp4", "Connected Components (Blobs)")
)

NameError: name 'show_media' is not defined